# 1A pilot v5 — follow-up checks (CPU only)

> **Tóm tắt (VI):** Notebook này **không chạy mô hình**. Nó đọc các file v4 và v5 trên Drive và trả lời hai câu hỏi còn mở trước khi viết bài. Chạy trên **CPU runtime** (`Runtime → Change runtime type → CPU`), `Run all`, cho phép Drive; mất vài phút. Kết quả ghi vào `1A-pilot-v5/followup/` và không ghi đè gì của v4 hay v5.

| # | Question | Why it matters for the paper |
|---|---|---|
| G1 | Is restoration worse on VSMEC and ViHSD because their **original text is already partly unaccented** (or slang), so that "errors" are measured against a wrong reference? | v5 scores the restorer against the original text. If the original lacks marks, a correct restoration counts as an error |
| G2 | Is the **negative Δ broken-syllables coefficient** (v4, item level) real? Where does it come from? | A sign against intuition must be explained or the RQ3 tokenizer claim is not safe |

**How G1 labels each word of the original text.** A reference vocabulary is built from **formal, edited text**: XNLI-vi and ViMMRC, test and validation splits. Each original word, lower-cased and without punctuation, gets one label:
- **accented**: it carries a diacritic;
- **missing mark**: it has no diacritic, but in the reference this spelling is almost always written with one. The reference must have ≥ 10 occurrences of its letters, and fewer than 2% of them may be unaccented. Examples: `khong` (the reference writes `không`), `nguoi`;
- **unknown**: it has no diacritic and never appears in the reference. These are slang, abbreviations, English words, typos and some names (`vl`, `ko`, `cmt`, `like`);
- **plain**: an ordinary Vietnamese word without a diacritic (`anh`, `em`, `cho`);
- **no letters**: punctuation, numbers and emoticons.

The labels are a proxy. For example, a name that the reference writes with marks can be labelled "missing mark".

## Setup and inputs

In [ ]:
import sys, os, re, json, unicodedata, collections, warnings
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
SMOKE = bool(os.environ.get("PILOT_SMOKE"))   # local test only
SRC4 = f"{ROOT}/1A-pilot-v4" + ("-smoke" if SMOKE else "")
SRC5 = f"{ROOT}/1A-pilot-v5" + ("-smoke" if SMOKE else "")
OUT = f"{SRC5}/followup"
for d in ["figures", "tables"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("reading", SRC4, "and", SRC5, "→ writing", OUT)

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)[:300]})
    print(("✅ " if ok else "❌ ") + name + (f" — {detail}" if detail else ""))
    pd.DataFrame(CHECKS).drop_duplicates("check", keep="last").to_csv(f"{OUT}/checks.csv", index=False)
    return ok

def require(name, ok, detail=""):
    if not check(name, ok, detail):
        raise RuntimeError(f"Stopped: {name}. {detail}")

TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
FIELDS = {"vsfc": ["sentence"], "vsmec": ["Sentence"], "vihsd": ["free_text"], "xnli": ["premise", "hypothesis"],
          "vimmrc": ["article", "question", "options"]}
WINDOW = 120   # the restorer's window in v5 (words per window)

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#2b2a28",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "pdf.fonttype": 42,
})
FIGS = []
def save(fig, name, caption):
    fig.savefig(f"{OUT}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT}/figures/{name}.pdf", bbox_inches="tight")
    FIGS.append((name, caption))
    plt.show()

def save_table(df, name, caption, floatfmt="%.3f"):
    df.to_csv(f"{OUT}/tables/{name}.csv")
    try:
        df.to_latex(f"{OUT}/tables/{name}.tex", float_format=floatfmt, caption=caption, label=f"tab:{name}", na_rep="--")
    except Exception as e:
        print("latex export failed:", name, repr(e)[:100])

def fmt_p(p):
    return "p < 1e-10" if p < 1e-10 else f"p = {p:.2g}"

def strip_diacritics(s):   # the same function as v4 and v5
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s)

KEY, LINES = {}, []   # → followup.json and followup.md

## G1 · Restoration errors and originally unaccented words

Inputs:
- v5 `data/test_items.csv`, which holds the original text (`clean`) and the restored text, item by item;
- v5's restoration-quality table. The first check recomputes its "words correct" column from the item file, so both use the same word split.

The reference text is downloaded from the Hugging Face parquet export, the same source v4 uses. If the download fails, the clean XNLI and ViMMRC items of v4 are used instead, and the notebook says so.

In [ ]:
D5 = pd.read_csv(f"{SRC5}/data/test_items.csv", keep_default_na=False)
RQ5 = pd.read_csv(f"{SRC5}/tables/table_restoration_quality.csv", index_col=0)
R = D5[D5.cond == "restored"].copy()
require("v5 item file has the restored condition for every task", set(R.task) == set(TASKS), sorted(set(R.task)))

PARQUET = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/{}/0000.parquet"
REF_SRC = {"xnli": (("facebook/xnli", "vi"), ["premise", "hypothesis"]), "vimmrc": (("ura-hcmut/ViMMRC", "default"), ["article", "question"])}
ref_texts, REF_FROM = [], "Hugging Face (XNLI-vi + ViMMRC, test + validation)"
try:
    for t, (src, cols) in REF_SRC.items():
        for split in ["test", "validation"]:
            d = pd.read_parquet(PARQUET.format(*src, split))
            for c in cols:
                ref_texts += list(d[c].drop_duplicates().astype(str))
except Exception as e:
    print("download failed:", repr(e)[:200])
    ref_texts = []
if not ref_texts:
    REF_FROM = "v4 clean items of XNLI-vi and ViMMRC (download failed)"
    for f in ["dev_items.csv", "test_items.csv"]:
        d = pd.read_csv(f"{SRC4}/data/{f}", keep_default_na=False)
        d = d[d.task.isin(["xnli", "vimmrc"])]
        ref_texts += list((d[d.cond == "clean"] if "cond" in d else d).text)
print("reference:", REF_FROM, "—", len(ref_texts), "texts")

def letters(w):
    return "".join(c for c in unicodedata.normalize("NFC", w) if c.isalpha()).lower()

ref = collections.Counter(letters(w) for x in ref_texts for w in x.split())
ref.pop("", None)
FORMS = collections.defaultdict(collections.Counter)
for w, n in ref.items():
    FORMS[strip_diacritics(w)][w] += n
check("reference vocabulary is large enough (≥ 5000 distinct words)", SMOKE or len(ref) >= 5000, f"{len(ref)} words")

def kind(word):
    w = letters(word)
    if not w:
        return "no letters"
    if strip_diacritics(w) != w:
        return "accented"
    forms = FORMS.get(w)
    if not forms:
        return "unknown"
    tot = sum(forms.values())
    return "missing mark" if tot >= 10 and forms.get(w, 0) / tot < 0.02 else "plain"
KINDS = ["accented", "plain", "missing mark", "unknown", "no letters"]

In [ ]:
# one row per word of the original text, aligned with the restored word (v5's split: lines, then single spaces)
rows = []
for r in R.itertuples():
    nf = len(FIELDS[r.task])
    for li, (a_line, b_line) in enumerate(zip(r.clean.split("\n"), r.text.split("\n"))):
        field = FIELDS[r.task][min(li, nf - 1)]
        a, b = a_line.split(" "), b_line.split(" ")
        for pos, (x, y) in enumerate(zip(a, b)):
            if x == "":
                continue
            rows.append((r.task, r.item_id, field, pos, x, y))
WD = pd.DataFrame(rows, columns=["task", "item_id", "field", "pos", "orig", "restored"])
WD["correct"] = WD.orig == WD.restored
WD["kind"] = WD.orig.map(kind)
WD["punct"] = WD.orig.map(lambda w: any(c.isalpha() for c in w) and not w.isalpha())   # punctuation glued to the word
WD["edge"] = (WD.pos % WINDOW < 5) | (WD.pos % WINDOW >= WINDOW - 5)                    # near a window boundary
WD["long_line"] = WD.groupby(["task", "item_id", "field"]).pos.transform("max") >= WINDOW

# the same word accuracy as v5 (v5 splits on any whitespace; the ViMMRC options lines give the same words)
acc5 = {}
for t, g in R.groupby("task"):
    tot = ok = 0
    for a, b in zip(g.clean, g.text):
        x, y = a.split(), b.split()
        tot += len(x); ok += sum(p == q for p, q in zip(x, y))
    acc5[t] = ok / tot
mine = WD.groupby("task").correct.mean()
check("word accuracy recomputed here equals v5's table", all(abs(acc5[t] - RQ5.loc[t, "words correct"]) < 1e-9 for t in TASKS)
      and all(abs(mine[t] - acc5[t]) < 1e-9 for t in TASKS),
      {t: (round(acc5[t], 4), round(mine[t], 4)) for t in TASKS})

In [ ]:
share = pd.crosstab(WD.task, WD.kind, normalize="index").reindex(index=TASKS, columns=KINDS).fillna(0)
save_table(share, "table_g1_word_kinds", f"Original test text: share of words by kind (reference: {REF_FROM}).")
print("share of the original words, by kind")
display(share.round(3))

err = WD[~WD.correct]
err_share = pd.crosstab(err.task, err.kind, normalize="index").reindex(index=TASKS, columns=KINDS).fillna(0)
print("share of the restoration errors, by kind of the original word")
display(err_share.round(3))

acc_kind = WD.pivot_table(index="task", columns="kind", values="correct", aggfunc="mean").reindex(index=TASKS, columns=KINDS)
G1 = pd.DataFrame({
    "words correct (v5)": WD.groupby("task").correct.mean(),
    "errors on missing-mark or unknown words": err.kind.isin(["missing mark", "unknown"]).groupby(err.task).mean(),
    "words correct, excluding missing-mark and unknown": WD[~WD.kind.isin(["missing mark", "unknown"])].groupby("task").correct.mean(),
    "accented words correct": acc_kind["accented"],
    "plain words correct": acc_kind["plain"],
    "missing-mark words given a mark": WD[WD.kind == "missing mark"].groupby("task").apply(lambda g: (g.restored != g.orig).mean()),
}).reindex(TASKS)
# for a missing-mark word, did the restorer write the form the reference uses most?
mm = WD[WD.kind == "missing mark"].copy()
mm["ref_form"] = mm.orig.map(lambda w: FORMS[letters(w)].most_common(1)[0][0])
mm["fixed"] = mm.restored.map(letters) == mm.ref_form
G1["missing-mark words restored to the reference form"] = mm.groupby("task").fixed.mean().reindex(TASKS)
save_table(G1, "table_g1_restoration_errors",
           "Restoration word accuracy against the original text, and how much of the error falls on words the original wrote without a mark (missing mark) or that are slang/foreign (unknown).")
display(G1.round(3))
KEY["g1"] = {"reference": REF_FROM, "share": share.to_dict("index"), "table": G1.to_dict("index")}

In [ ]:
# other places where the restorer could fail: glued punctuation, window boundaries, long lines, ViMMRC fields
parts = []
for name, col in [("punctuation glued to the word", "punct"), ("within 5 words of a window boundary", "edge"),
                  ("line longer than one window", "long_line")]:
    g = WD[WD.kind.isin(["accented", "plain"])].groupby(["task", col]).correct.mean().unstack()
    g.columns = [f"{name}: {'yes' if c else 'no'}" for c in g.columns]
    parts.append(g)
G1b = pd.concat(parts, axis=1).reindex(TASKS)
save_table(G1b, "table_g1_where_errors", "Restoration word accuracy (accented and plain words only) by position and form of the word.")
print("word accuracy on accented + plain words only")
display(G1b.round(3))
G1c = WD[(WD.task == "vimmrc") & WD.kind.isin(["accented", "plain"])].groupby("field").correct.agg(["mean", "size"])
save_table(G1c, "table_g1_vimmrc_fields", "ViMMRC: restoration word accuracy (accented and plain words) per field.")
display(G1c.round(3))
KEY["g1"]["where"] = G1b.to_dict("index")
KEY["g1"]["vimmrc_fields"] = G1c["mean"].to_dict()

# per item: restoration accuracy against the share of missing-mark + unknown words
it = WD[WD.kind != "no letters"].groupby(["task", "item_id"]).agg(acc=("correct", "mean"),
                                                                      informal=("kind", lambda k: k.isin(["missing mark", "unknown"]).mean()))
it["bin"] = pd.cut(it.informal, [-1e-9, 0, 0.1, 1], labels=["0", "(0, 10%]", "> 10%"])
G1d = it.groupby(["task", "bin"], observed=False).acc.agg(["mean", "size"]).unstack("bin").reindex(TASKS)
save_table(G1d, "table_g1_by_informal_share", "Item-level restoration word accuracy by the share of missing-mark and unknown words in the original item.")
display(G1d.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
ax = axes[0]
bottom = np.zeros(len(TASKS))
COL = {"accented": "#2a78d6", "plain": "#9fc1ea", "missing mark": "#eb6834", "unknown": "#eda100", "no letters": "#c9c8c2"}
for k in KINDS:
    ax.bar(TASKS, share[k].values, bottom=bottom, color=COL[k], label=k, edgecolor="white", lw=1)
    bottom += share[k].values
ax.set(ylabel="share of original words", title="Kinds of words in the original text")
ax.legend(fontsize=7, ncol=2)
ax = axes[1]
x = np.arange(len(TASKS))
ax.bar(x - 0.2, G1["words correct (v5)"], 0.4, color="#8a8984", label="all words (v5)")
ax.bar(x + 0.2, G1["words correct, excluding missing-mark and unknown"], 0.4, color="#2a78d6", label="without missing-mark and unknown")
ax.set_xticks(x, TASKS)
ax.set(ylim=(0.5, 1), ylabel="restoration word accuracy", title="Restoration accuracy against the original")
ax.legend(fontsize=7, loc="lower left")
plt.tight_layout()
save(fig, "figG1_restoration_errors", "Left: kinds of words in the original test text. Right: restoration word accuracy on all words and without the words the original wrote without a mark or that are slang/foreign.")

## G2 · The negative Δ broken-syllables coefficient

Input: v4's `tables/item_level.csv.gz`. It has one row per (model, task, item, noisy condition) where the model was right on the clean item and the noise changed the item. The outcome is `flip`, i.e. the model becomes wrong. The model is a linear probability model with item × condition and model fixed effects, with standard errors clustered by unit, as in v4.

`d_broken` is the share of syllables split into more than one token, on the noisy text minus on the clean text. A negative coefficient means: when noise splits more syllables, the model flips *less*.

Checks:
1. **The sign of Δ broken itself.** Which tokenizers break more and which break fewer syllables when diacritics are removed?
2. **Is the sign stable** across tasks and conditions? These rows come from v4's own table.
3. **Decomposition.** `d_broken` = broken(noisy) − broken(clean). Do the two parts have opposite effects of equal size, as the difference assumes? Or does one part carry the sign?
4. **Leave one out.** Refit without each tokenizer group and without each model family. Is one group driving the sign?
5. **Another definition.** Δ broken syllables counted per *clean* syllable, (n_broken_noisy − n_broken_clean) / n_syllables_clean, like `d_tok`.
6. **Shape.** Binned plot, after removing the fixed effects.

In [ ]:
X = pd.read_csv(f"{SRC4}/tables/item_level.csv.gz")
FE4 = pd.read_csv(f"{SRC4}/tables/table_item_level_fe.csv")
X = X.dropna(subset=["flip", "d_nll", "d_tok", "d_broken"]).copy()
X["broken_c"] = X.nbroken_c / X.nsyl_c
X["broken_n"] = X.nbroken / X.nsyl.clip(lower=1)
X["d_broken_cnt"] = (X.nbroken - X.nbroken_c) / X.nsyl_c
require("item-level file has every column G2 needs", {"unit", "tok_class", "family", "item_cond"} <= set(X.columns))
print(len(X), "rows,", X.model.nunique(), "models,", X.tok_class.nunique(), "tokenizer groups")

def demean(df, cols, groups, iters=200, tol=1e-9):
    out = df[cols].astype(float).copy()
    for _ in range(iters):
        delta = 0
        for gcol in groups:
            mns = out.groupby(df[gcol].values).transform("mean")
            out -= mns
            delta = max(delta, float(np.abs(mns.to_numpy()).max()))
        if delta < tol:
            break
    return out

def fe(df, xs):
    # v4's model: item×condition and model fixed effects, predictors in SD units, clustered by unit
    df = df[df.item_cond.map(df.item_cond.value_counts()) > 1]
    if len(df) < 50 or df.model.nunique() < 2:
        return None
    dm = demean(df, ["flip"] + xs, ["item_cond", "model"])
    sd = df[xs].std().replace(0, np.nan)
    keep = [x for x in xs if np.isfinite(sd[x])]
    r = sm.OLS(dm.flip, dm[keep] / sd[keep]).fit(cov_type="cluster", cov_kwds={"groups": df.unit})
    return pd.DataFrame({"pp_per_SD": 100 * r.params, "lo": 100 * r.conf_int()[0], "hi": 100 * r.conf_int()[1], "p": r.pvalues,
                         "rows": len(df)})

base = fe(X, ["d_nll", "d_tok", "d_broken"])
v4row = FE4[(FE4["sample"] == "all valid models") & (FE4.term == "d_broken")]
check("the v4 coefficient is reproduced", len(v4row) == 1 and abs(base.loc["d_broken", "pp_per_SD"] - v4row.pp_per_SD.iloc[0]) < 1e-6,
      f"here {base.loc['d_broken', 'pp_per_SD']:.3f} pp, v4 {v4row.pp_per_SD.iloc[0]:.3f} pp" if len(v4row) else "v4 row missing")

In [ ]:
# 1 · the sign of Δ broken per tokenizer group
fam_of = X.groupby("tok_class").family.agg(lambda s: ", ".join(sorted(set(s))))
G2a = X[X.cond.isin(["partial50", "strip", "teencode"])].pivot_table(index="tok_class", columns="cond", values="d_broken", aggfunc="mean")
G2a.insert(0, "broken share, clean", X.groupby("tok_class").broken_c.mean())
G2a.insert(0, "families", fam_of)
save_table(G2a, "table_g2_broken_by_tokenizer", "Share of clean syllables split into more than one token, and its change under noise, per tokenizer group.")
display(G2a.round(3))

# 2 · stability in v4's own table
G2b = FE4[FE4.term == "d_broken"].set_index("sample")[["pp_per_SD", "lo", "hi", "p", "rows"]]
display(G2b.round(4))

# 3 · decomposition, 4 · leave one out, 5 · another definition
rows = []
def add(name, df, xs, terms):
    r = fe(df, xs)
    if r is None:
        return
    for k in terms:
        if k in r.index:
            rows.append({"model": name, "term": k, **r.loc[k].to_dict()})
add("v4: Δ NLL + Δ tokens + Δ broken", X, ["d_nll", "d_tok", "d_broken"], ["d_nll", "d_tok", "d_broken"])
add("parts: broken clean and broken noisy", X, ["d_nll", "d_tok", "broken_c", "broken_n"], ["broken_c", "broken_n"])
add("parts, without Δ NLL and Δ tokens", X, ["broken_c", "broken_n"], ["broken_c", "broken_n"])
add("Δ broken + broken clean", X, ["d_nll", "d_tok", "d_broken", "broken_c"], ["d_broken", "broken_c"])
add("Δ broken per clean syllable", X, ["d_nll", "d_tok", "d_broken_cnt"], ["d_broken_cnt"])
for c in ["partial50", "strip", "teencode"]:
    add(f"parts, condition {c}", X[X.cond == c], ["d_nll", "d_tok", "broken_c", "broken_n"], ["broken_c", "broken_n"])
G2c = pd.DataFrame(rows)
save_table(G2c.set_index(["model", "term"]), "table_g2_decomposition",
           "Item-level fixed-effects LPM (pp per SD): Δ broken syllables split into its clean and noisy parts, and another definition.")
display(G2c.round(4))

loo = []
for col in ["tok_class", "family"]:
    for g in sorted(X[col].unique()):
        r = fe(X[X[col] != g], ["d_nll", "d_tok", "d_broken"])
        if r is not None:
            loo.append({"left out": f"{col} = {g}", **r.loc["d_broken"].to_dict()})
G2d = pd.DataFrame(loo).set_index("left out")
save_table(G2d, "table_g2_leave_one_out", "Δ broken-syllables coefficient (pp per SD, v4 model) with one tokenizer group or one model family left out.")
display(G2d.round(4))
KEY["g2"] = {"by_tokenizer": G2a.drop(columns="families").to_dict("index"), "v4_rows": G2b.to_dict("index"),
             "decomposition": G2c.to_dict("records"), "leave_one_out": G2d.to_dict("index")}

In [ ]:
# 6 · shape: flip rate against Δ broken after removing the fixed effects (and Δ NLL, Δ tokens)
Xf = X[X.item_cond.map(X.item_cond.value_counts()) > 1]
dm = demean(Xf, ["flip", "d_nll", "d_tok", "d_broken"], ["item_cond", "model"])
res = lambda y: y - sm.OLS(y, dm[["d_nll", "d_tok"]]).fit().fittedvalues
yb, xb = res(dm.flip), res(dm.d_broken)
q = pd.qcut(xb, 10, duplicates="drop")
bins = pd.DataFrame({"x": xb, "y": yb}).groupby(q, observed=True).agg(x=("x", "mean"), y=("y", "mean"), n=("y", "size"))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
ax = axes[0]
ax.plot(bins.x, 100 * bins.y, "o-", color="#1baf7a")
ax.axhline(0, color="#8a8984", lw=0.8)
ax.set(xlabel="Δ broken syllables (residual)", ylabel="flip rate (pp, residual)",
       title="Flip rate by decile of Δ broken (FE, Δ NLL and Δ tokens removed)")
ax = axes[1]
d = G2c[G2c.model.str.startswith("parts:") | G2c.model.str.startswith("v4:")]
labels = [f"{m.split(':')[0]} · {t}" for m, t in zip(d.model, d.term)]
ax.errorbar(d.pp_per_SD, range(len(d)), xerr=[d.pp_per_SD - d.lo, d.hi - d.pp_per_SD], fmt="o", color="#2a78d6", capsize=3)
ax.axvline(0, color="#8a8984", lw=0.8)
ax.set_yticks(range(len(d)), labels)
ax.set(xlabel="pp flip per SD (95% CI)", title="Δ broken and its two parts")
plt.tight_layout()
save(fig, "figG2_broken", "Left: flip rate by decile of Δ broken syllables, after the fixed effects and the other predictors are removed. Right: coefficients of Δ broken and of its clean and noisy parts.")

## Conclusions (generated from the numbers above)

In [ ]:
L = ["# 1A pilot v5 · follow-up checks", "", f"Reference vocabulary for G1: {REF_FROM}.", "", "## G1 · Restoration errors"]
g = G1
L.append("| Task | Words correct (v5) | Missing-mark + unknown words in the original | Share of errors on them | Words correct without them |")
L.append("|---|---|---|---|---|")
for t in TASKS:
    L.append(f"| {t} | {g.loc[t, 'words correct (v5)']:.3f} | {share.loc[t, 'missing mark'] + share.loc[t, 'unknown']:.3f} | "
             f"{g.loc[t, 'errors on missing-mark or unknown words']:.3f} | {g.loc[t, 'words correct, excluding missing-mark and unknown']:.3f} |")
social, formal = ["vsmec", "vihsd"], ["vsfc", "xnli"]
gap_all = g.loc[formal, "words correct (v5)"].mean() - g.loc[social, "words correct (v5)"].mean()
gap_ex = (g.loc[formal, "words correct, excluding missing-mark and unknown"].mean()
          - g.loc[social, "words correct, excluding missing-mark and unknown"].mean())
explained = 1 - gap_ex / gap_all if gap_all > 0 else np.nan
KEY["g1"]["gap"] = {"all": gap_all, "excluding": gap_ex, "explained": explained}
L += ["", f"- The accuracy gap between the formal tasks (VSFC, XNLI) and the social-media tasks (VSMEC, ViHSD) is {100 * gap_all:.1f} pp "
          f"on all words and {100 * gap_ex:.1f} pp without the missing-mark and unknown words. "
          f"These words therefore account for about {100 * explained:.0f}% of the gap."]
if not gap_all > 0:
    L.append("- There is no gap to explain: restoration is not less accurate on the social-media tasks.")
elif explained >= 0.7:
    L.append("- **The hypothesis holds:** most of the lower accuracy on VSMEC and ViHSD comes from words the original text wrote without marks, or from slang and foreign words. This is a property of the reference, not of the restorer.")
elif explained >= 0.3:
    L.append("- **The hypothesis holds only in part:** these words explain part of the gap. The restorer is also worse on ordinary words of informal text.")
else:
    L.append("- **The hypothesis does not hold:** the gap remains when these words are left out. The restorer is worse on informal text itself.")
given = g["missing-mark words given a mark"]
L.append("- Of the missing-mark words, the restorer gave a mark to " + ", ".join(f"{t} {given[t]:.0%}" for t in TASKS) + " (counted as errors in v5).")
L.append(f"- ViMMRC, accented + plain words only: {', '.join(f'{k} {v:.3f}' for k, v in G1c['mean'].items())}. "
         f"Long lines (> {WINDOW} words) vs short ones: see `table_g1_where_errors`.")

L += ["", "## G2 · Δ broken syllables"]
b = G2c.set_index(["model", "term"])
v4b = b.loc[("v4: Δ NLL + Δ tokens + Δ broken", "d_broken")]
pc, pn = b.loc[("parts: broken clean and broken noisy", "broken_c")], b.loc[("parts: broken clean and broken noisy", "broken_n")]
L.append(f"- v4 coefficient reproduced: {v4b.pp_per_SD:+.2f} pp per SD ({fmt_p(v4b.p)}).")
signs = G2b.pp_per_SD
L.append(f"- In v4's own table the sign is negative in {int((signs < 0).sum())} of {len(signs)} samples; "
         f"significant (p < 0.05) and negative in {int(((signs < 0) & (G2b.p < 0.05)).sum())}, significant and positive in {int(((signs > 0) & (G2b.p < 0.05)).sum())}.")
L.append(f"- Split into parts: broken share on the clean text {pc.pp_per_SD:+.2f} pp per SD ({fmt_p(pc.p)}); "
         f"on the noisy text {pn.pp_per_SD:+.2f} pp ({fmt_p(pn.p)}).")
sig = lambda r: r.p < 0.05
if sig(pc) and pc.pp_per_SD > 0 and not (sig(pn) and pn.pp_per_SD > 0):
    L.append("- **Reading:** the negative Δ comes from the *clean* part. Where the clean text is already split into many pieces, the model flips more; "
             "Δ broken is low there because there is little left to break. So Δ broken works as a proxy for poor tokenization of the clean text, not for damage done by the noise.")
elif sig(pn) and pn.pp_per_SD < 0:
    L.append("- **Reading:** more broken syllables on the *noisy* text go with fewer flips, even with the clean level held fixed. "
             "The effect sits in the noisy text. One possible reason is that a syllable split into pieces still exposes its known sub-word parts. This needs an explanation in the paper.")
else:
    L.append("- **Reading:** neither part has a clear effect of the expected kind, so Δ broken should be reported as having no reliable effect on its own.")
lo_sign = G2d.pp_per_SD
L.append(f"- Leave one out: the coefficient stays negative in {int((lo_sign < 0).sum())} of {len(lo_sign)} refits "
         f"(range {lo_sign.min():+.2f} to {lo_sign.max():+.2f} pp). Largest change: without {(lo_sign - v4b.pp_per_SD).abs().idxmax()}.")
alt = b.loc[("Δ broken per clean syllable", "d_broken_cnt")]
L.append(f"- Counted per clean syllable: {alt.pp_per_SD:+.2f} pp per SD ({fmt_p(alt.p)}).")
TEXT = "\n".join(L)
open(f"{OUT}/followup.md", "w").write(TEXT)
display(Markdown(TEXT))

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(f"{float(o):.4g}")
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o
json.dump(jsonable(KEY), open(f"{OUT}/followup.json", "w"), ensure_ascii=False, indent=1)
pd.DataFrame(FIGS, columns=["figure", "caption"]).to_csv(f"{OUT}/figures/captions.csv", index=False)
print("checks:", sum(c["ok"] for c in CHECKS), "passed,", sum(not c["ok"] for c in CHECKS), "failed")
print("Everything is in", OUT, "— when done: Runtime → Disconnect and delete runtime.")